# Feature value, not feature count
## OTTO · controlled historical-demand and family ablations

**Fitting-only research. No selection/evaluation access, feature promotion, or Kaggle submission.**

This notebook replays committed measurements; it never trains a model. The candidate-relative hypothesis was tested, not assumed to help.

In [1]:
from pathlib import Path
import hashlib, json, math
from IPython.display import display, HTML
candidates = [Path.cwd(), *Path.cwd().parents]
root = next(p for p in candidates if (p / "reports/research/feature_value_pilot.json").is_file())
path = root / "reports/research/feature_value_pilot.json"
assert hashlib.sha256(path.read_bytes()).hexdigest() == "db38587e9bba57b4fd28990a79f768b9c77120a52509827eee14bb2ae2274341"
report = json.loads(path.read_text())
assert not report["protocol"]["selection_access"]
assert not report["protocol"]["evaluation_access"]
print("Verified committed evidence; source", report["source"]["commit"][:12])

Verified committed evidence; source b9f53c29297a


In [2]:
weights = (0.1, 0.3, 0.6)
for name, arm in report["arms"].items():
    recomputed = sum(w*h/d for w,h,d in zip(weights, arm["hits"], arm["denominators"]))
    assert math.isclose(recomputed, arm["weighted_recall_at_20"], abs_tol=1e-12)
    assert [sum(x) for x in zip(*arm["fold_hits"])] == arm["hits"]
print("All nine pooled scores and fold hit totals reconcile.")
rows = "".join(f"<tr><td>{name}</td><td>{arm['features']}</td><td>{arm['weighted_recall_at_20']:.6f}</td><td>{arm['fold_scores'][0]:.6f}</td><td>{arm['fold_scores'][1]:.6f}</td></tr>" for name,arm in report["arms"].items())
display(HTML("<table><thead><tr><th>Arm</th><th>Features</th><th>Pooled Recall@20</th><th>Earlier fold</th><th>Later fold</th></tr></thead><tbody>"+rows+"</tbody></table>"))

All nine pooled scores and fold hit totals reconcile.


Arm,Features,Pooled Recall@20,Earlier fold,Later fold
baseline,102,0.470574,0.351240,0.645175
baseline_relative,114,0.468411,0.362149,0.624142
shared,134,0.484737,0.379714,0.637782
shared_relative,146,0.486757,0.368805,0.658047
without_episode,128,0.478679,0.369611,0.637419
without_funnel,123,0.473916,0.356786,0.644218
without_graph_degree,132,0.486992,0.373058,0.652421
without_graph_raw,131,0.479461,0.373058,0.634180
without_graph_row,124,0.490712,0.389110,0.638592


In [3]:
import plotly.graph_objects as go
ordered = sorted(report["arms"], key=lambda k: report["arms"][k]["weighted_recall_at_20"])
fig = go.Figure(go.Bar(x=[report["arms"][k]["weighted_recall_at_20"] for k in ordered], y=ordered, orientation="h", text=[f"{report['arms'][k]['weighted_recall_at_20']:.4f}" for k in ordered], textposition="outside"))
fig.update_layout(template=None, title="Matched fitting-only feature comparisons", xaxis_title="Pooled weighted Recall@20 (not a leaderboard score)", height=460, margin=dict(l=200,r=90,t=70,b=70), xaxis=dict(range=[0,0.56]))
fig.show(renderer="plotly_mimetype")

## Interpretation

The popularity-relative family failed the stability condition: adding it to the baseline regressed overall, and adding it to the shared representation improved only slightly with opposite signs across folds. It is not promoted.

Dropping row-normalized graph features improves both fold point estimates, but uncertainty still includes regression. That is a controlled follow-up hypothesis, not proof that these features are universally harmful. Funnel, episode, and raw-graph removal lower the pooled point estimate.

Only **92 order-denominator units** support the temporal validation. The folds are dependent, the query prefixes are retrospective, and intervals are descriptive. The historical winning private score **0.60503** is not comparable to these development values.

In [4]:
for key in ("shared_minus_baseline", "baseline_relative_minus_baseline", "shared_relative_minus_shared", "shared_minus_without_graph_row"):
    result = report["comparisons"][key]
    print(key, "delta:", round(result["difference"], 6), "fold deltas:", [round(x,6) for x in result["fold_differences"]], "95% descriptive interval:", [round(x,6) for x in result["descriptive_95_interval"]])
print("Models:", report["execution"]["new_models"], "| replay fits:", report["execution"]["replay_new_models"])
print("Training experiment seconds:", round(report["execution"]["first_pass_seconds"],3), "| peak MiB:", round(report["execution"]["peak_rss_mib"],2))

shared_minus_baseline delta: 0.014163 fold deltas: [0.028474, -0.007393] 95% descriptive interval: [-0.012741, 0.039994]
baseline_relative_minus_baseline delta: -0.002163 fold deltas: [0.010909, -0.021033] 95% descriptive interval: [-0.020001, 0.013984]
shared_relative_minus_shared delta: 0.00202 fold deltas: [-0.010909, 0.020265] 95% descriptive interval: [-0.015465, 0.022754]
shared_minus_without_graph_row delta: -0.005975 fold deltas: [-0.009396, -0.00081] 95% descriptive interval: [-0.019238, 0.004751]
Models: 54 | replay fits: 0
Training experiment seconds: 60.49 | peak MiB: 1390.68


## Next research gate

Reuse the completed models and feature checkpoints. Evaluate missing-target coverage from broader action-conditioned and multi-hop retrieval before spending on another large ranker run. Confirm the row-normalized ablation on a larger fitting-only cohort with the same controls. The separate rolling as-of-demand family remains untested on real snapshots.

Primary references: [official metric](https://www.kaggle.com/competitions/otto-recommender-system), [winning approach](https://www.kaggle.com/competitions/otto-recommender-system/writeups/mrkmakr-1st-place-solution), [historical leaderboard](https://www.kaggle.com/competitions/otto-recommender-system/leaderboard).

**Feature engineering remains open. No record-beating claim is made.**